# Comparison of SVM and Deep Learning Models
Performance metrics on the test set and spatio-temporal performance analysis

In [ ]:
import warnings
import pandas as pd
import numpy as np
import h3
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.svm import SVR, LinearSVR, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    mean_absolute_percentage_error, mean_squared_error
)
import geopandas as gpd
import contextily as ctx
import shapely
import ast
import torch
import torch.nn.functional as F
import joblib
import torch.nn as nn

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODEL_DIR = Path("../models")
VISUALS_DIR = Path("../plots")
TARGET = "Total_Trip_Start" # Target variable for prediction
df_test_8 = pd.read_parquet("../data/prediction_split/res_8/df_test.parquet")
df_test_7 = pd.read_parquet("../data/prediction_split/res_7/df_test.parquet")

In [2]:
# Feature columns for the different models
BASIC_FEATURES = [
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday", "day_of_week",
    "lat", "lon", "scikit_distance_to_loop", "base_demand"
]
POI_COLS = ['poi_cat_automotive','poi_cat_civic_community', 'poi_cat_education', 'poi_cat_entertainment',
            'poi_cat_finance', 'poi_cat_food_drink', 'poi_cat_grocery', 'poi_cat_health', 'poi_cat_leisure_sports', 
            'poi_cat_lodging','poi_cat_nightlife', 'poi_cat_services', 'poi_cat_shopping','poi_cat_transport']
WEATHER = ["2m_temp_c", "total_precip_mm", "snow_cov", "snow_depth", "wind_speed"]

feature_sets = {
    "basic": BASIC_FEATURES,
    "basic+poi": BASIC_FEATURES + POI_COLS,
    "basic+weather": BASIC_FEATURES + WEATHER,
    "basic+poi+weather": BASIC_FEATURES + POI_COLS + WEATHER,
}

In [13]:
# Load all models from the models directory
model_files = list(MODEL_DIR.glob("*.joblib"))
performance_metrics = []
prediction_7_df = pd.DataFrame()
prediction_8_df = pd.DataFrame()

# Inspect the loaded models
for model_file in model_files:
    model = joblib.load(model_file)
    if "7" in model_file.stem:
        test_df = df_test_7
    elif "8" in model_file.stem:
        test_df = df_test_8
    y_test = test_df[TARGET].values
    # Find feature set based on feature elements
    if set(model['features']) == set(feature_sets["basic"]):
        feature_set = "basic"
    elif set(model['features']) == set(feature_sets["basic+poi"]):
        feature_set = "basic+poi"
    elif set(model['features']) == set(feature_sets["basic+weather"]):
        feature_set = "basic+weather"
    elif set(model['features']) == set(feature_sets["basic+poi+weather"]):
        feature_set = "basic+poi+weather"
    else:
        feature_set = "unknown"
    
    print(f"Loaded model: {model_file.name}")
    print(f"Model Hyperparameters: {model['model']}")
    print(f"Target Value: {model['target']}")
    print(f"Features Count: {len(model['features'])}")
    print(f"Feature Set: {feature_set}")

    # Model Evaluation on test set
    X_test = test_df[model['features']].values
    y_pred = model['model'].predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred, squared=False)
    if "7" in model_file.stem:
        prediction_7_df["y_pred" + f"_{model_file.stem}"] = y_pred
    elif "8" in model_file.stem:
        prediction_8_df["y_pred" + f"_{model_file.stem}"] = y_pred
    else:
        print(f"Unknown Spatial Resolution: {model_file.name}")
    
    print(f"MAE: {mae:.4f}, R2: {r2:.4f}, RMSE: {rmse:.4f}")
    print("-" * 40)
    # Store the metrics in the DataFrame
    performance_metrics.append({
        "model_file": model_file.name,
        "model_type": type(model['model']).__name__,
        "feature_set": feature_set,
        "resolution": "7" if "7" in model_file.stem else "8",
        "mae": mae,
        "r2": r2,
        "rmse": rmse,
    })

Loaded model: svr_best_8.joblib
Model Hyperparameters: SVR(C=10, cache_size=500)
Target Value: Total_Trip_Start
Features Count: 30
Feature Set: basic+poi+weather
Metrics: {'Model': 'Tuned SVR (rbf)', 'MAE': 0.3621372993602783, 'RMSE': 2.6653662445591193, 'R2': 0.8439693872709221}
----------------------------------------
Loaded model: svr_best.joblib
Model Hyperparameters: SVR(C=50, cache_size=500)
Target Value: Total_Trip_Start
Features Count: 25
Feature Set: basic+poi
Metrics: {'Model': 'Tuned SVR (rbf)', 'MAE': 0.34491972823373696, 'RMSE': 2.4996575058542883, 'R2': 0.8627674788880126}
----------------------------------------
Loaded model: svr_best_7.joblib
Model Hyperparameters: SVR(C=50, cache_size=500, kernel='poly')
Target Value: Total_Trip_Start
Features Count: 25
Feature Set: basic+poi
Metrics: {'Model': 'Tuned SVR (poly)', 'MAE': 2.4348391367357123, 'RMSE': 16.039788101835654, 'R2': 0.37125012032018734}
----------------------------------------
Loaded model: svr_best_8_no_poi_no

In [ ]:
# Erneute instanziierung von FNN, damit aus den .pth die modelle erstellt werden können
class FNN(nn.Module):
    def __init__(self, input_size, hidden_layers):
        super(FNN, self).__init__()
        self.layers = nn.ModuleList()
        prev_size = input_size
        for hidden_size in hidden_layers:
            self.layers.append(nn.Linear(prev_size, hidden_size))
            prev_size = hidden_size

        self.output_layer = nn.Linear(prev_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.output_layer(x).squeeze(1)
    
FNN_FEATURES = [
    "lat", "lon", "scikit_distance_to_loop",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "day_of_week", "is_weekend", "is_holiday", "is_near_holiday",
    "2m_temp_c", "total_precip_mm", "snow_cov", "snow_depth", "wind_speed",
    "poi_cat_automotive", "poi_cat_civic_community", "poi_cat_education",
    "poi_cat_entertainment", "poi_cat_finance", "poi_cat_food_drink",
    "poi_cat_grocery", "poi_cat_health", "poi_cat_leisure_sports",
    "poi_cat_lodging", "poi_cat_nightlife", "poi_cat_services",
    "poi_cat_shopping", "poi_cat_transport"
]

In [ ]:
model_files = [f for f in MODEL_DIR.glob("*.pth") if "res7" in f.stem or "res8" in f.stem]

performance_metrics = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for model_file in model_files:
    stem = model_file.stem

    if "res7" in stem:
        resolution = "7"
        test_df = df_test_7
    else:
        resolution = "8"
        test_df = df_test_8

    # get metadata from the filename
    hidden_layers = ast.literal_eval(stem.split("layers=")[1].split("_lr=")[0])
    lr = float(stem.split("_lr=")[1])

    # Load the model
    checkpoint = torch.load(model_file, map_location=device)
    model = FNN(input_size=len(FNN_FEATURES), hidden_layers=hidden_layers)
    model.load_state_dict(checkpoint)
    model.to(device)
    model.eval()

    # Prediction on test-split
    X_test = torch.tensor(test_df[FNN_FEATURES].values, dtype=torch.float32).to(device)
    y_test = test_df[TARGET].values

    with torch.no_grad():
        y_pred = np.clip(np.expm1(model(X_test).cpu().numpy().flatten()), 0, None)

    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"Model: {model_file.name}")
    print(f"Layers: {hidden_layers} | LR: {lr}")
    print(f"MAE: {mae:.4f} | R²: {r2:.4f} | RMSE: {rmse:.4f}")
    print("-" * 50)

    performance_metrics.append({
        "model_file"   : model_file.name,
        "model_type"   : "FNN",
        "hidden_layers": str(hidden_layers),
        "lr"           : lr,
        "resolution"   : resolution,
        "mae"          : mae,
        "r2"           : r2,
        "rmse"         : rmse,
    })

Model: best_fnn_res7_layers=[256, 128, 64, 32]_lr=0.001.pth
Layers: [256, 128, 64, 32] | LR: 0.001
MAE: 166507707.0406 | R²: -676114419714243584.0000 | RMSE: 16632968625.7883
--------------------------------------------------
Model: best_fnn_res8_layers=[128, 64, 32, 16]_lr=0.001.pth
Layers: [128, 64, 32, 16] | LR: 0.001
MAE: 0.7034 | R²: -0.0104 | RMSE: 6.7825
--------------------------------------------------


In [ ]:
performance_metrics

## Performance Metrics Table
Here we compare our performance metrics on the test set for multiple SVM and NN models

In [14]:
# Convert the performance metrics list to a DataFrame
performance_df = pd.DataFrame(performance_metrics)
performance_df = performance_df.sort_values(by="model_file").reset_index(drop=True)
performance_df

,model_file,model_type,feature_set,mae,r2,rmse
0,svr_best.joblib,SVR,basic+poi,0.344920,0.862767,2.499658
1,svr_best_7.joblib,SVR,basic+poi,2.434839,0.371250,16.039788
2,svr_best_8.joblib,SVR,basic+poi+weather,0.362137,0.843969,2.665366
3,svr_best_8_no_poi_no_weather.joblib,SVR,basic,0.343935,0.853410,2.583474


In [ ]:
# Visualize the R2 for all models in a bar plot
plt.figure(figsize=(12, 6))
sns.barplot(data=performance_df, x="model_file", y="r2", hue="feature_set")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.title("Model Performance (R2) by Model File and Feature Set")   
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Inter Model Prediction Comparison 
## Spatio Performance Analytics

## 9. Spatial prediction quality
The aggregate metrics above collapse the whole city into a single number and hide strong **spatial** variation in how well the model performs. Here we visualize the SVR predictions on the test set and aggregate the error by H3 cell. 
Two complementary views answer "where does the model work?":

- **MAE per hexagon** — the absolute size of the prediction miss (trips/hour).
- **R² per hexagon** — how much of each cell's hour-to-hour demand *variation* the
  model explains. Cells whose demand is essentially flat (the quiet outskirts)
  have no variation to explain, so their R² is undefined and they are greyed out.

In [15]:
# Community area outlines, used as a spatial reference overlay in the choropleths.
community_areas = gpd.read_file(Path("..") / "data" / "chicago_community_areas.gpkg")

def plot_hexagon_performance(hex_perf_gdf, value_col, title):
    """
    Plot the per-hexagon performance metrics on a choropleth map.
    """
    fig, ax = create_choropleth_plot(
        hex_perf_gdf, value_col=value_col, title=title,
        boundaries=community_areas, cmap="YlGnBu", figsize=(8, 10), k=7
    )
    return fig, ax

def get_hexagon_performance_metrics(df_eval, pred_col, target_col=TARGET):
    """
    Compute performance metrics for each hexagon based on the evaluation DataFrame.
    """
    df_eval["abs_err"] = (df_eval[target_col] - df_eval[pred_col]).abs()
    df_eval["sq_err"] = (df_eval[target_col] - df_eval[pred_col]) ** 2

    grp = df_eval.groupby("h3_index")
    hex_perf = grp.agg(
        n_obs=(target_col, "size"),
        mean_actual=(target_col, "mean"),
        var_actual=(target_col, "var"),
        mae=("abs_err", "mean"),
        mse=("sq_err", "mean"),
    )
    hex_perf["rmse"] = np.sqrt(hex_perf["mse"])
    ss_res = grp["sq_err"].sum()
    ss_tot = hex_perf["var_actual"].fillna(0) * (hex_perf["n_obs"] - 1)
    hex_perf["r2"] = 1 - ss_res / ss_tot.replace(0, np.nan)
    hex_perf["rel_error"] = hex_perf["mae"] / hex_perf["mean_actual"].replace(0, np.nan)
    hex_perf = hex_perf.drop(columns=["var_actual", "mse"]).reset_index()
    hex_perf_gdf = get_hexagon_grid(hex_perf, h3_col="h3_index")
    return  hex_perf, hex_perf_gdf

def get_hexagon_grid(data, h3_col="h3_index"):
    """
    Build a GeoDataFrame of H3 hexagon polygons, preserving all columns from the source.
    """
    def _to_polygon(cell):
        boundary = h3.cell_to_boundary(cell)
        return shapely.geometry.Polygon([(lng, lat) for lat, lng in boundary])

    if isinstance(data, pd.Series):
        h3_list = list(data)
        df = pd.DataFrame({"h3_index": h3_list})
    else:
        h3_list = list(data[h3_col])
        df = data.reset_index(drop=True)
    geometries = [_to_polygon(c) for c in h3_list]
    return gpd.GeoDataFrame(df, geometry=geometries, crs="EPSG:4326")


def create_choropleth_plot(gdf, value_col="poi_count", title="", boundaries=None,
                           cmap="YlOrRd", figsize=(6, 8), k=7):
    """
    Create a static matplotlib choropleth of hexagon values with Chicago basemap.
    """
    if boundaries is None:
        boundaries = community_areas
    fig, ax = plt.subplots(figsize=figsize)
    gdf.to_crs(epsg=3857).plot(
        ax=ax, column=value_col, cmap=cmap, scheme="quantiles", k=k,
        edgecolor="none", alpha=0.8, legend=True, zorder=1,
        legend_kwds=dict(title=value_col.replace("_", " ").title(),
                         loc="lower right", fontsize=8, title_fontsize=9),
        missing_kwds=dict(color="lightgrey", label="No data"),
    )
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, zoom=11)
    boundaries.to_crs(epsg=3857).boundary.plot(
        ax=ax, color="#222", linewidth=1.0, alpha=0.9, zorder=3)
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()
    return fig, ax

def generate_summarizing_map_plot(hex_perf_gdf, figsize=(10, 16)):
    fig, ax = plt.subplots(figsize=figsize)
    # add three plots for R2, MSE, and RMSE next to eachother
    ax1 = plt.subplot(3, 1, 1)
    plot_hexagon_performance(hex_perf_gdf, value_col="r2", title=f"Model Performance (R2) by Hexagon - {model_file}")
    ax2 = plt.subplot(3, 1, 2)
    plot_hexagon_performance(hex_perf_gdf, value_col="rel_error", title=f"Model Performance (rel_error) by Hexagon - {model_file}")
    ax3 = plt.subplot(3, 1, 3)
    plot_hexagon_performance(hex_perf_gdf, value_col="rmse", title=f"Model Performance (RMSE) by Hexagon - {model_file}")
    return fig, ax


_IncompleteInputError: incomplete input (837107431.py, line 79)

In [ ]:
# Find best performing model_file based on R2 score by model_type - only resolution 7 models are considered here
res_7_models = performance_df[performance_df["resolution"] == "7"]
best_models = res_7_models.loc[res_7_models.groupby("model_type")["r2"].idxmax()]["model_file"]
for model_file in best_models:
    print(f"Best model for {model_file}:")
    print(res_7_models[res_7_models["model_file"] == model_file])
    print("-" * 40)
    pred_col = "y_pred" + f"_{model_file.stem}"
    hex_perf, hex_perf_gdf = get_hexagon_performance_metrics(prediction_8_df, pred_col=pred_col, target_col=TARGET)
    generate_summarizing_map_plot(hex_perf_gdf, figsize=(10, 16))


# Find best performing model_file based on R2 score by model_type - only resolution 8 models are considered here
res_8_models = performance_df[performance_df["resolution"] == "8"]
best_models = res_8_models.loc[res_8_models.groupby("model_type")["r2"].idxmax()]["model_file"]
for model_file in best_models:
    print(f"Best model for {model_file}:")
    print(res_8_models[res_8_models["model_file"] == model_file])
    print("-" * 40)
    pred_col = "y_pred" + f"_{model_file.stem}"
    hex_perf, hex_perf_gdf = get_hexagon_performance_metrics(prediction_8_df, pred_col=pred_col, target_col=TARGET)
    generate_summarizing_map_plot(hex_perf_gdf, figsize=(10, 16))

### Map 1: mean absolute error (MAE) per hexagon
This is the absolute size of the prediction miss. Because demand itself is far higher downtown, the largest absolute errors concentrate in the Loop; the quiet outskirts have tiny MAE simply because there is little demand.

In [ ]:
#| label: fig-svm-hex-mae
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='mae',
    title='SVR absolute error per hexagon (MAE, trips/hour)\nred = larger prediction error',
    cmap='YlOrRd',
    k=7,
)
plt.show()

### Map 2: mean relative error (MRE) per hexagon
This is the relative size of the prediction miss. 

In [ ]:
#| label: fig-svm-hex-mre
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='rel_error',
    title='SVR relative error per hexagon (MRE, trips/hour)\nred = larger prediction error',
    cmap='YlOrRd',
    k=7,
)
plt.show()

### Map 3: coefficient of determination (R2) per hexagon.
R2 measures how much of each cell's hour-to-hour demand *variation* the model explains. Cells whose demand never moves (flat ~0 in the outskirts) have no variance to explain -> R2 is undefined and they are drawn grey ("No data"). Green = the temporal demand pattern is captured well; red = poorly.

In [ ]:
#| label: fig-svm-hex-r2
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='r2',
    title='SVR prediction quality per hexagon (R²)\ngreen = well predicted, grey = too little demand to assess',
    cmap='RdYlGn',
    k=7,
)
plt.show()

## Performance Visualization

### SVR - Support Vector Regression

In [ ]:
# fig-svm-hex-mae - MAE per hexagon for the best SVR model
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='mae',
    title='SVR absolute error per hexagon (MAE, trips/hour)\nred = larger prediction error',
    cmap='YlOrRd',
    k=7,
)
plt.show()

In [ ]:
# fig-svm-hex-mre - MRE per hexagon for the best SVR model
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='rel_error',
    title='SVR relative error per hexagon (MRE, trips/hour)\nred = larger prediction error',
    cmap='YlOrRd',
    k=7,
)
plt.show()

In [ ]:
# fig-svm-hex-r2 - R² per hexagon for the best SVR model
fig, ax = create_choropleth_plot(
    hex_perf_gdf,
    value_col='r2',
    title='SVR prediction quality per hexagon (R²)\ngreen = well predicted, grey = too little demand to assess',
    cmap='RdYlGn',
    k=7,
)
plt.show()

### Deep Learning - Neural Network

# Time Performance Analysis

In [ ]:
# On which days of the week does the models perform best/worst?


In [ ]:
# How does the model perform throughout the day?

In [ ]:
#| label: fig-svm-pred-vs-actual
# Predicted vs actual for the best model
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best, alpha=0.2, s=8, color="royalblue")
lim = y_test.max()
plt.plot([0, lim], [0, lim], "r--", lw=2)
plt.title(f"{best_row}: Actual vs. Predicted Demand")
plt.xlabel("Actual Trip Count")
plt.ylabel("Predicted Trip Count")
plt.tight_layout()
plt.show()